# Transformer Foundations, Part 5A: Why Cross-Attention Exists

## Guiding question

Why keep every source position if one pooled sentence vector is cheaper?

Decoder-only generation uses one growing tape. Translation, summarization, and structured transformation have two workspaces: read a complete source, then write a distinct target. The reversal diagnostic makes the required routing visible:

```text
source:  [3, 1, 4, 1]
target:  [1, 4, 1, 3]
```

| Complaint | Refinement | Evidence |
|---|---|---|
| One mean vector forgets which occurrence was where | Preserve one encoder vector per source position | Reordered sources collide under mean pooling |
| Decoder state and source memory have different roles | Q from decoder; K/V from encoder | Weight grid has shape `(target, source)` |
| Teacher-forced token scores look strong | Run free-running decoding | One early error changes every later prefix |
| Repeated source/prefix work is wasteful | Separate fixed and growing caches | Encoder and cross-KV stay fixed; self-KV grows |

Keep [the handwritten theory companion](05-encoder-decoder-and-cross-attention-theory.md) beside this notebook. Part 5B implements the reversal microscope.


In [ ]:
# Imports and deterministic routing example
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

np.random.seed(505)
source = np.array([3, 1, 4, 1])
target = source[::-1]
print("source:", source.tolist())
print("target:", target.tolist())


## 1. Attempt 1: squeeze the source into one mean

**Predict:** Can the mean distinguish `[3, 1, 4, 1]` from `[1, 4, 1, 3]`?

Mean pooling keeps content totals and discards arrangement. Adding one position vector to each token before taking the mean still sums the same set of positions, so the collision remains.


In [ ]:
# Measured bottleneck: two different orders collapse to the same summary
token_vectors = {
    1: np.array([1.0, 0.0, 0.2]),
    3: np.array([0.0, 1.0, 0.4]),
    4: np.array([0.2, 0.3, 1.0]),
}
position_vectors = np.eye(4, 3)

def pooled(sequence):
    states = np.stack(
        [token_vectors[token] + position_vectors[index] for index, token in enumerate(sequence)]
    )
    return states.mean(axis=0)

source_pool = pooled(source)
reordered_pool = pooled(target)
print("source mean:   ", np.round(source_pool, 4))
print("reordered mean:", np.round(reordered_pool, 4))
print("maximum difference:", float(np.abs(source_pool - reordered_pool).max()))
assert np.allclose(source_pool, reordered_pool)


#### The mean knows what appeared, but not which address to retrieve

Reversal needs a different source address at every target step. The smallest repair is not a larger pooled vector. It is an addressable memory with one row per source position.

## 2. Cross-attention keeps the roles separate

- decoder states form **queries**: what does this writing step need?
- encoder states form **keys**: what does each source position advertise?
- encoder states also form **values**: what information is delivered if selected?

The resulting table is `(target_length, source_length)`. Those lengths need not match. Cross-attention is not causally masked because the complete source is legal; it still needs a source padding mask.


In [ ]:
# Hand-built routing scores: each target step asks for the reversed source address
source_positions = np.arange(len(source))
target_steps = np.arange(len(target))
desired_positions = len(source) - 1 - target_steps
scores = -4.0 * (target_steps[:, None] * 0 + source_positions[None, :] - desired_positions[:, None]) ** 2
weights = np.exp(scores - scores.max(axis=-1, keepdims=True))
weights = weights / weights.sum(axis=-1, keepdims=True)

fig, ax = plt.subplots(figsize=(6, 4))
image = ax.imshow(weights, vmin=0, vmax=1, cmap="magma")
ax.set_xticks(source_positions, [f"src {i}: {token}" for i, token in enumerate(source)])
ax.set_yticks(target_steps, [f"step {i}: {token}" for i, token in enumerate(target)])
ax.set_xlabel("encoder key/value position")
ax.set_ylabel("decoder query position")
ax.set_title("Expected reversal routing")
fig.colorbar(image, ax=ax, label="attention weight")
plt.tight_layout()
plt.show()

routed = weights.argmax(axis=-1)
print("predicted source addresses:", routed.tolist())
assert np.array_equal(routed, desired_positions)


In [ ]:
# Animation: the anti-diagonal appears one decoder query at a time
fig, ax = plt.subplots(figsize=(6, 4))
shown = np.zeros_like(weights)
image = ax.imshow(shown, vmin=0, vmax=1, cmap="magma")
ax.set_xticks(source_positions, [str(value) for value in source])
ax.set_yticks(target_steps, [str(value) for value in target])
ax.set_xlabel("source token")
ax.set_ylabel("target token")

def update(frame):
    shown[frame] = weights[frame]
    image.set_data(shown)
    ax.set_title(f"decoder step {frame} retrieves source position {desired_positions[frame]}")
    return [image]

animation = FuncAnimation(fig, update, frames=len(target), interval=700, blit=False)
plt.close(fig)
display(HTML(animation.to_jshtml()))


## 3. Teacher forcing creates another honest-looking shortcut

During training:

```text
decoder input:  [BOS, y0, y1, y2, ...]
expected output:[y0,  y1, y2, y3, ..., EOS]
```

All target positions are scored in parallel, and causal self-attention stops a position from reading later gold tokens. During generation, however, the decoder receives its own earlier choices. One wrong token changes the later query states. This mismatch is exposure bias.

Teacher-forced token accuracy answers, "Can the model predict with a correct history?" Free-running sequence accuracy answers, "Can it finish after conditioning on itself?" Report both.


In [ ]:
# Prefix lineage: one early mistake contaminates every later free-running input
gold = ["<bos>", "1", "4", "1", "3"]
free = ["<bos>", "1", "3", "1", "3"]
print("step | teacher-forced prefix | free-running prefix")
for step in range(1, len(gold)):
    teacher_prefix = " ".join(gold[:step])
    free_prefix = " ".join(free[:step])
    marker = "same" if teacher_prefix == free_prefix else "changed"
    print(f"{step:>4} | {teacher_prefix:<23} | {free_prefix:<23} ({marker})")


## 4. Three reusable states, not one universal cache

| Reusable state | Built from | Lifetime |
|---|---|---|
| Encoder memory | complete source | fixed for the request |
| Cross-attention K/V | layer-specific projections of encoder memory | fixed for the request |
| Decoder self-attention K/V | generated target prefix | grows one position per step |

Fixed encoder activations during inference do not mean frozen encoder parameters during training. Target loss normally backpropagates through cross-attention into the encoder.

## Part 5A checkpoint

**Built and measured:** mean-pooling collision, rectangular `(target, source)` routing, expected reversal anti-diagonal, prefix contamination.

**Explained or illustrated:** teacher forcing, exposure bias, fixed encoder and cross-KV caches, growing self-KV cache.

**Named and deferred:** beam search and production T5/BART demonstrations; they do not change the routing contract.

**Unresolved question:** will a learned `CrossAttention` module recover the predicted route and still generate without gold history? Continue to [Part 5B](05b-encoder-decoder-and-cross-attention-lab.ipynb).
